# CS 195: Natural Language Processing
## Chat and Instruct Models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericmanley/s26-CS195NLP/blob/main/F2_1_ChatInstruct.ipynb)

## References


* [Hugging Face Chat Basics](https://huggingface.co/docs/transformers/en/conversations)
* [SmolLM2: When Smol Goes Big — Data-Centric Training of a Small Language Model](https://arxiv.org/pdf/2502.02737)


## Demo Day

Sit with the same people you sat with last week

* Each person do a 5-min demo of **creative synthesis** project or completed **applied exploration** (or **core practice** if that's what you have)
* Write down the names of the people you presented to (you'll include this in your portfolio later)
* (optional) Nominate a cool project to show off to everyone



## Install Modules

We'll be using `transformers` version 5. You probably only need to run this if you are doing this for the first time on your own computer. If so, uncomment these two lines and run it.


In [ ]:
import sys
!{sys.executable} -m pip install transformers accelerate

## Large vs. Small language models

Large Language Models (GPT 5.2, Claude Opus 4.6, Grok 4.1, Gemini 3) get all the attention.

Smaller language models have come a long way too, and they require much less computation
* can often be run on a laptop or a Colab instance
* can be *fine-tuned* for specific applications with good performance


### Example: SmolLM2

Hugging Face developed a [family of small language models called SmolLM](https://huggingface.co/collections/HuggingFaceTB/smollm2) there's also a [SmolLM3](https://huggingface.co/blog/smollm3)

SmolLM2 comes in various sizes
* 135M (135 million parameters - weights in the neural network)
* 360M
* 1.7B

Contrast with the LLMs above which likely all have over 100 billion parameters and run on a cluster of devices in a data center

Each size has a **base model**, like [SmolLM2-360M](https://huggingface.co/HuggingFaceTB/SmolLM2-360M)
* *pre-trained* on lots of diverse text
* designed to predict the *next word* - it's your phone's keyboard text prediction on steroids

The *base model* is then fine-tuned on *instruction following* and *conversational data*, which make it useful for building **chat bots**.
* resulting model has a name like [SmolLM2-360M-Instruct](https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct)

## Building a chat bot with the `text-generation` pipeline

Setting up a chat bot works the same way as other Hugging Face models we've seen, but we'll use the `text-generation` pipeline

In [4]:
from transformers import pipeline
from accelerate import Accelerator

device = Accelerator().device

chatbot = pipeline("text-generation", model="HuggingFaceTB/SmolLM2-360M-Instruct", device = device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

### Chat template

*Instruct* models often allow you to pass the input in using a **Chat Template** like this

In [6]:
chat_history = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Explain gravity in one paragraph."},
]

Now lets get the response and display what is returned

In [7]:
response = chatbot(chat_history)
response

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': [{'role': 'system',
    'content': 'You are a helpful assistant.'},
   {'role': 'user', 'content': 'Explain gravity in one paragraph.'},
   {'role': 'assistant',
    'content': 'Gravity is a fundamental force of nature that affects all objects with mass or energy. It is a universal force that attracts two objects with mass towards each other, causing them to move towards each other. This attraction is described by the law of universal gravitation, which states that every point mass attracts every other point mass by a force acting along the line intersecting both points. The force of gravity is proportional to the product of the masses of the two objects and inversely proportional to the square of the distance between their centers. This means the stronger the gravitational force between two objects, the more they pull towards each other.'}]}]

**Notice:**
* it just uses normal lists and dictionaries
* it returns the entire chat history on the `'generated_text'` key

If you wanted to just print out the response, you could do it like this

In [ ]:
print("ASSISTANT:", response[0]['generated_text'][-1]['content'])

ASSISTANT: Gravity is a fundamental force of nature that attracts two masses towards each other, causing objects to fall to the ground, and influencing the orbits of celestial bodies. It is characterized by its inverse square law, which states that the force of gravity between two objects is proportional to the product of their masses and inversely proportional to the square of the distance between them. This means that objects with more mass and closer proximity to each other experience greater gravitational forces.


## Multi-turn conversations

If you want to have a multi-turn conversation, you need to pass *the entire chat history* to the model - it doesn't have any inherent memory of the output it just gave you

In [ ]:
another_chat_history = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Explain gravity in one paragraph."},
    {"role": "assistant", "content": "Gravity is a fundamental force of nature that causes objects with mass to attract each other. According to Sir Isaac Newton, the force of gravity between two objects depends on their masses and the distance between them. The larger the mass of the objects, the greater the gravitational pull. This force is responsible for keeping planets in orbit around the sun and causes objects to fall towards the ground when dropped. Albert Einstein also described gravity as a curvature of spacetime caused by massive objects, which in turn warps the fabric of spacetime around them."},
    {"role": "user", "content": "Which of those two do you think has had a bigger impact on the field?"}
]

next_response = chatbot(another_chat_history)
print(next_response[0]['generated_text'][-1]['content'])

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


According to many researchers, the theory of general relativity has had a more significant impact on the field of physics and astronomy than Newton's law of universal gravitation. General relativity describes gravity as the curvature of spacetime caused by the presence of mass and energy, which has far-reaching implications for our understanding of the universe, from the behavior of black holes to the expansion of the cosmos.


## Exercise

Write a loop that allows for back-and-forth conversation with the model. Make sure to keep track of the full history of the chat as you go.

In [ ]:
from IPython.display import HTML, display

def set_css():
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))
get_ipython().events.register('pre_run_cell', set_css)

chat_history1 = [
    {"role": "system", "content": "You are a helpful assistant."}
]

while True:
  user_input = input("You: ")
  if user_input.lower() == "stop":
    break

  chat_history1.append({"role": "user", "content": user_input})

  response = chatbot(chat_history1)
  assistant_response = response[0]['generated_text'][-1]['content']

  print("Assistant:", assistant_response)
  chat_history1.append({"role": "assistant", "content": assistant_response})

You: hi


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Assistant: Hello! How can I help you today?
You: what is the best strategy for turn-based games?


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Assistant: For turn-based games, the best strategy is to focus on maintaining a steady hand and managing your actions. This can help you avoid overextending or overextending yourself.


KeyboardInterrupt: Interrupted by user

## Evaluating Chat Models: Benchmarks

A benchmark is a dataset with one or more reference answers that can be used to measure a model's response (like the reference summaries we compared against with ROUGE)


Model benchmarking is a big deal - companies like to report how well their models perform on all kinds of benchmarks

For example, see the **performance** tab here: https://deepmind.google/models/gemini/pro/

Take a look at this benchmark for multi-turn conversations: https://huggingface.co/datasets/HuggingFaceH4/mt_bench_prompts

**Group Discussion:** What are some things you notice about this data?



## Evaluating Chat Models: Human Evaluators

Language models are often evaluated by having humans perform A/B testing where two models respond to the same prompt, and the human indicates which was better.

Try it out here: https://arena.ai

## Group Exercise

Do the following as a group:
* Come up with 5 language model prompts - what are some questions/instructions you think would help you decide how good a language model is?
* Test them using [SmolLM2-360M-Instruct](https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct) and [Qwen/Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct)
* Have each person in your group vote on which one they thought was the best
* Write down the results

## Applied Exploration

Choose two instruct models of similar size: https://huggingface.co/models?pipeline_tag=text-generation&sort=trending&search=instruct
  * Link to the model cards for the models you're using and describe each of them

Do one of the following:

1. Go to https://huggingface.co/datasets and find a dataset suitable to use as a benchmark, and compare the performance of the two models. It doesn't have to be a conversational benchmark - it could be a text classification, summarization, math, etc. dataset, as long as you can instruct the model to answer it. And, you don't have to use the whole dataset.
    * link to and describe the dataset
    * describe how you compared the performance (e.g., what metric did you use?)
    * report the results

OR

2. Come up with your own fun benchmark (Taylor Swift trivia, AI Dungeon Master, Joke telling, etc.), generate responses for both models, and have another person rate the answers.
    * Describe what you did
    * report the results


Models:
1. https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct
2. https://huggingface.co/RedHatAI/Llama-3.2-1B-Instruct-FP8




In [8]:
from IPython.display import HTML, display

def set_css():
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))
get_ipython().events.register('pre_run_cell', set_css)

chatbot1 = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct", device = device)
chat_history = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Write a short story that involves a tiefling, a hafling, and an elf all meeting each other in a tavern at the start of their adventure."},
]

response = chatbot1(chat_history)
response

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': [{'role': 'system',
    'content': 'You are a helpful assistant.'},
   {'role': 'user',
    'content': 'Write a short story that involves a tiefling, a hafling, and an elf all meeting each other in a tavern at the start of their adventure.'},
   {'role': 'assistant',
    'content': 'Once upon a time, in the heart of the old world, there was a small village nestled between two mountains. The villagers were known for their hospitality and their love of stories. One day, as they gathered around the fire to tell tales, a group of travelers entered the village.\n\nThe first traveler was a tall man with dark skin and piercing eyes. He wore a tattered cloak and carried a large axe. "We have come from far away," he said, his voice deep and gravelly. "We seek a place where we can rest and gather information about our journey."\n\nNext came a smaller figure with curly hair and freckles. She had a mischievous smile and a curious gaze. "I am a Hafling," she declared, holding up

In [9]:
from IPython.display import HTML, display

def set_css():
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))
get_ipython().events.register('pre_run_cell', set_css)

chatbot2 = pipeline("text-generation", model="allenai/OLMo-2-0425-1B-Instruct", device = device)
chat_history = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Write a short story that involves a tiefling, a hafling, and an elf all meeting each other in a tavern at the start of their adventure."},
]

response = chatbot2(chat_history)
response

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.97G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/179 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/581 [00:00<?, ?B/s]

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': [{'role': 'system',
    'content': 'You are a helpful assistant.'},
   {'role': 'user',
    'content': 'Write a short story that involves a tiefling, a hafling, and an elf all meeting each other in a tavern at the start of their adventure.'},
   {'role': 'assistant',
    'content': 'Once upon a time in the bustling city of Silvermoon, amidst the grand spires and stately homes, there was a small, yet lively tavern known as The Elden\'s Brew. It was a quaint establishment, with wooden tables draped in soft draperies and walls adorned with tapestries that told tales of ancient battles and enchanted quests.\n\nOn a fine morning, Taelon, a tiefling of fiery ambition and indomitable spirit, wandered into The Elden\'s Brew. His scales shimmered like molten gold under the barkeep\'s gentle torchlight, casting an aura of warmth and determination. Beside him, a halfling named Fennic hopped in, his pointy ears perking with curiosity, and a cheerful twinkle in his eyes. Finally

Model 1 response: Once upon a time, in the heart of the old world, there was a small village nestled between two mountains. The villagers were known for their hospitality and their love of stories. One day, as they gathered around the fire to tell tales, a group of travelers entered the village.The first traveler was a tall man with dark skin and piercing eyes. He wore a tattered cloak and carried a large axe. "We have come from far away," he said, his voice deep and gravelly. "We seek a place where we can rest and gather information about our journey." Next came a smaller figure with curly hair and freckles. She had a mischievous smile and a curious gaze. "I am a Hafling," she declared, holding up her hands. "And I am here to help you find your way through this strange land." Then there was a young girl with long, flowing hair. Her eyes sparkled with wonder and curiosity. "I am an Elf," she announced proudly. "And I know many secrets of the forest." As the three newcomers settled into the cozy inn, the hostess smiled warmly at them. "Welcome, friends! To make you feel at home, I will prepare you a hearty meal and some ale.

Model 2 response: Once upon a time in the bustling city of Silvermoon, amidst the grand spires and stately homes, there was a small, yet lively tavern known as The Elden's Brew. It was a quaint establishment, with wooden tables draped in soft draperies and walls adorned with tapestries that told tales of ancient battles and enchanted quests. On a fine morning, Taelon, a tiefling of fiery ambition and indomitable spirit, wandered into The Elden's Brew. His scales shimmered like molten gold under the barkeep's gentle torchlight, casting an aura of warmth and determination. Beside him, a halfling named Fennic hopped in, his pointy ears perking with curiosity, and a cheerful twinkle in his eyes. Finally, a slyly dressed elf named Ithrin glided in, her cobalt gaze scanning the room for allies or potential adversaries.The barkeep, an elderly human with a beard as white as the moonlight and eyes that twinkled with kindness, set down his jug of the finest elven ale and gestured for the newcomers to take a seat."Taelon, Fennic, and Ithrin," he introduced, a knowing smile playing on his lips

## Reflection
I asked the models to come up with a short story that takes place in a fantasy setting in order to test its creativity. Overall, the people I asked about the responses preferred the second response because the first one felt unrealistic to them. Even in fantasy settings, people usually introduce themselves by their name and not their race/species.